# 言語資源のクレンジング・重複排除と PII マスキング

書籍 第5章「言語資源のつくり方」（5-1 言語資源の入手方法、5-4 コーパス作成、練習問題 1）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、コーパス構築時に不可欠な前処理・プライバシー保護・品質チェックを体験します。

1. **テキストのクレンジングと正規化**: HTML タグ除去、Unicode 正規化（NFKC）、余分な空白の統一
2. **完全重複・近似重複（ニアデュープリケート）の検出**: 文字 n-gram の Jaccard 係数による類似文書の特定
3. **データリーク実験**: 重複を除去せずに訓練/テスト分割すると評価スコアがどれだけ過大評価されるかを確認
4. **PII（個人情報）マスキングと品質フィルタリング（5-1・練習問題 1）**: メールアドレス・電話番号・郵便番号など、所定の形式に合う文字列の置換と、文字種比率・長さに基づくスパム除外

## 1. テキストのクレンジングと Unicode 正規化（5-4）

Web クロールなどで集めたテキストには、HTML タグ、全角英数字・記号の表記揺れ（`ＡＩ` vs `AI`、`ｶﾀｶﾅ` vs `カタカナ`）、連続空白などが混在します。

In [1]:
import html
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)                 # HTMLタグ除去
    text = re.sub(r"https?://\S+", " ", text)            # URL除去
    text = unicodedata.normalize("NFKC", text)           # Unicode正規化（全角英数→半角、半角カナ→全角等）
    text = re.sub(r"\s+", " ", text).strip()             # 空白の統一
    return text

dirty_samples = [
    "<p>ＮＬＰの教科書（２０２４年）を読みました！</p> https://example.com",
    "ﾃｽﾄ環境で　　BERT＆GPTを動かした。",
]
for s in dirty_samples:
    print("前:", s)
    print("後:", clean_text(s), "\n")

前: <p>ＮＬＰの教科書（２０２４年）を読みました！</p> https://example.com
後: NLPの教科書(2024年)を読みました! 

前: ﾃｽﾄ環境で　　BERT＆GPTを動かした。
後: テスト環境で BERT&GPTを動かした。 



---

## 2. 文字 n-gram の Jaccard 係数による近似重複検出

完全一致だけでなく、文末の記号や 1〜2 語だけが異なる近似重複（ニアデュープリケート）を検出するため、文字 3-gram 集合の **Jaccard 係数**（$|A \cap B| / |A \cup B|$）を計算します。

In [2]:
def char_trigrams(text):
    t = clean_text(text)
    return set(t[i:i + 3] for i in range(max(1, len(t) - 2)))

def jaccard(a, b):
    sa, sb = char_trigrams(a), char_trigrams(b)
    return len(sa & sb) / len(sa | sb)

s1 = "駅前で美味しいラーメンを食べました。"
s2 = "駅前で美味しいラーメンを食べました！"      # 記号違い（近似重複）
s3 = "駅前のレストランでパスタを食べました。"      # 別文書

print(f"s1 vs s2 (近似重複): Jaccard = {jaccard(s1, s2):.3f}")
print(f"s1 vs s3 (別文書)  : Jaccard = {jaccard(s1, s3):.3f}")

s1 vs s2 (近似重複): Jaccard = 0.882
s1 vs s3 (別文書)  : Jaccard = 0.179


---

## 3. 重複排除を怠ったときのデータリーク実験

Web から集めたレビューデータに「同じ内容のテンプレート投稿やコピペ（近似重複）」が多く含まれている状況を模擬します。
**重複を除去せずにランダム分割**すると、訓練データとほぼ同じ文がテストデータにも紛れ込み（データリーク）、未知の文に対する性能より高い正解率が出ることがあります。

In [3]:
rng = np.random.default_rng(42)

# それぞれ表現が異なるユニークな文（pos 20件, neg 20件）を作成
pos_topics = [
    "部屋の清掃が隅々まで行き届いていて気持ちよく過ごせました",
    "温泉の大浴場が広くて旅の疲れがしっかり取れました",
    "朝食の焼きたてパンと地元野菜のサラダがとても美味しかったです",
    "スタッフの笑顔と丁寧な案内のおかげで安心して滞在できました",
    "駅から歩いてすぐの立地で大きな荷物があっても便利でした",
    "ベッドの寝心地が良く朝までぐっすり眠ることができました",
    "静かな環境でオンライン会議や仕事にも集中できました",
    "チェックイン前の荷物預かりに快く対応してもらい助かりました",
    "窓からの夜景が美しく記念日の良い思い出になりました",
    "アメニティが充実していて手ぶらで泊まれて快適でした",
    "館内のWi-Fiが高速で動画視聴もストレスがありませんでした",
    "貸切風呂の予約が取りやすく家族でゆっくり入浴できました",
    "夕食の郷土料理がボリューム満点で大変満足しました",
    "駐車場が無料で広いため車での旅行でも安心でした",
    "空気清浄機や加湿器が部屋に完備されていて快適でした",
    "ラウンジの無料コーヒーサービスが嬉しかったです",
    "防音がしっかりしており廊下の音も気になりませんでした",
    "子供の宿泊向けサービスが丁寧で家族連れに優しい宿でした",
    "価格以上の設備とサービスでコストパフォーマンスが高いです",
    "また機会があればぜひリピートしたいと思えるホテルでした",
]
neg_topics = [
    "部屋の隅にホコリが残っていて清掃状態に不満がありました",
    "大浴場の脱衣所が狭く混雑していて落ち着きませんでした",
    "朝食の品数が少なく補充も遅かったのが残念でした",
    "フロントの対応が事務的で質問しても素っ気ない印象でした",
    "駅から遠く坂道も多いため徒歩での移動が大変でした",
    "マットレスが硬すぎて夜中に何度も目が覚めてしまいました",
    "外の車の騒音が夜遅くまで響いてよく眠れませんでした",
    "チェックインの手続きに30分以上待たされて疲れました",
    "建物の古さが目立ち水回りの設備に使いづらさを感じました",
    "部屋の照明が暗くデスクで作業するには不便でした",
    "Wi-Fiの接続が頻繁に切れて仕事になりませんでした",
    "エアコンの温度調節が効かず部屋が暑いままでした",
    "壁が薄く隣の部屋の話し声が丸聞こえで気になりました",
    "エレベーターが1基しかなく朝の移動に時間がかかりました",
    "シャワーの水圧が弱くお湯の温度も安定しませんでした",
    "駐車場の案内が分かりにくく周辺を何度も回りました",
    "エレベーター前の自動販売機が故障したままでした",
    "予約時のリクエストが伝わっておらず確認に時間がかかりました",
    "内容を考えると宿泊料金がやや割高に感じました",
    "改善されない限り次回の利用は控えようと思います",
]

unique_texts = pos_topics + neg_topics
unique_labels = [1] * len(pos_topics) + [0] * len(neg_topics)

# コーパスに「近似重複（末尾の記号違いや再投稿）」が混入している状況を模擬
leaked_texts = []
leaked_labels = []
for idx, (t, y) in enumerate(zip(unique_texts, unique_labels)):
    leaked_texts.append(t)
    leaked_labels.append(y)
    if idx % 2 == 0:  # 半分の文がテンプレート的に 5 倍に重複しているとする
        for suffix in ["。", "！", "♪", "（再投稿）"]:
            leaked_texts.append(t + suffix)
            leaked_labels.append(y)

print(f"重複ありコーパス: {len(leaked_texts)} 件")

# A) 重複を除去せずそのまま分割（データリークあり）
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    leaked_texts, leaked_labels, test_size=0.25, stratify=leaked_labels, random_state=42
)

# B) Jaccard >= 0.75 を近似重複とみなして重複排除してから分割（クリーン）
dedup_texts, dedup_labels = [], []
for t, y in zip(leaked_texts, leaked_labels):
    if not any(jaccard(t, kept) >= 0.75 for kept in dedup_texts):
        dedup_texts.append(t)
        dedup_labels.append(y)

print(f"重複排除後コーパス: {len(dedup_texts)} 件")
X_tr_clean, X_te_clean, y_tr_clean, y_te_clean = train_test_split(
    dedup_texts, dedup_labels, test_size=0.25, stratify=dedup_labels, random_state=42
)

def eval_clf(X_tr, y_tr, X_te, y_te):
    clf = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3)), LogisticRegression(max_iter=1000))
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

acc_leak = eval_clf(X_tr_leak, y_tr_leak, X_te_leak, y_te_leak)
acc_clean = eval_clf(X_tr_clean, y_tr_clean, X_te_clean, y_te_clean)

pd.DataFrame([
    {"実験条件": "A. 重複排除なし（データリークあり）", "データ件数": len(leaked_texts), "テスト正解率": round(acc_leak, 3)},
    {"実験条件": "B. 近似重複排除後", "データ件数": len(dedup_texts), "テスト正解率": round(acc_clean, 3)},
])

重複ありコーパス: 120 件
重複排除後コーパス: 40 件


,実験条件,データ件数,テスト正解率
0,A. 重複排除なし（データリークあり）,120,0.9
1,B. 近似重複排除後,40,0.5


条件 A では、訓練文とほぼ同じ文がテスト側に混ざります。この実験では、重複排除によって訓練・テストの件数も変わるため、正解率の差には件数や分割の影響も含まれます。未知の文への汎化を測る際は、分割前に重複を除くか、同じ原文から作られた文や近似重複を同じ分割にまとめます。

---

## 4. PII（個人情報）マスキングと品質フィルタリング（5-1・5-4・練習問題 1）

第 5 章 練習問題 1 でも触れているとおり、コーパス構築時には著作権だけでなく **個人情報やセンシティブなデータの保護** が不可欠です。また、Web クロールデータには極端に短い断片、他言語スパム、記号羅列などのゴミ文も多く含まれます。

ここでは正規表現による **PII マスキング** と、**日本語比率・記号比率・文字長に基づくルールベース品質フィルタ** を実装して動作確認します。置換の対象は正規表現に合う形式です。例えば、表の2行目には郵便番号を置換した後も地名が残ります。実データを扱う際は、置換後のテキストも点検してください。

In [4]:
def mask_pii(text):
    """個人情報・連絡先をプレースホルダへ置換（長いパターン・境界を先に処理して誤爆を防ぐ）"""
    t = unicodedata.normalize("NFKC", text)
    t = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[EMAIL]", t)
    t = re.sub(r"(?<!\d)(?:\d{4}[-\s]?){3}\d{4}(?!\d)", "[CARD_OR_ID]", t)
    t = re.sub(r"(?<!\d)(?:\+81[-\s]?|0\d{1,4}-\d{1,4}-\d{3,4}|0(?:70|80|90)\d{8})(?!\d)", "[PHONE]", t)
    t = re.sub(r"(?:〒\s*\d{3}-\d{4}|(?<![\d-])\d{3}-\d{4}(?![\d-]))", "[POSTAL]", t)
    t = re.sub(r"@[A-Za-z0-9_]{2,}", "[USER]", t)
    return t

def text_quality_metrics(text):
    body = clean_text(re.sub(r"\[(?:EMAIL|POSTAL|PHONE|CARD_OR_ID|USER)\]", "", mask_pii(text)))
    non_ws = re.sub(r"\s+", "", body)
    n = len(non_ws)
    if n == 0:
        return {"len": 0, "ja_ratio": 0.0, "sym_ratio": 1.0, "clean_body": body}
    ja_chars = len(re.findall(r"[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF、。]", non_ws))
    sym_chars = len(re.findall(r"[^\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFFA-Za-z0-9、。]", non_ws))
    return {
        "len": n,
        "ja_ratio": ja_chars / n,
        "sym_ratio": sym_chars / n,
        "clean_body": body,
    }

def quality_filter(text, min_len=8, max_len=400, min_ja_ratio=0.35, max_sym_ratio=0.30):
    m = text_quality_metrics(text)
    if m["len"] < min_len:
        return False, "too_short", m
    if m["len"] > max_len:
        return False, "too_long", m
    if m["ja_ratio"] < min_ja_ratio:
        return False, "low_ja_ratio", m
    if m["sym_ratio"] > max_sym_ratio:
        return False, "high_symbol_ratio", m
    return True, "ok", m

raw_web_samples = [
    "駅近で部屋も清潔でした。問い合わせは support.hotel@example.co.jp または 03-1234-5678 まで。",
    "荷物の発送先は 〒191-0065 東京都日野市... 連絡は 090-9876-5432 (@tanaka_travel) へ！",
    "会員ID 1234-5678-9012-3456 で宿泊手続きを完了しました。とてもスムーズでした。",
    "https://spam-link.example.com/promo?id=999 Click here!!",
    "Best hotel deal in Tokyo! Book now and get 50% OFF today only!",
    "おすすめ★★★★★!!!!!ホテル予約%%%%%今すぐ",
    "okです",
    "<p>朝食のパンが焼きたてで美味しく、スタッフの笑顔も素敵でした。</p>",
]

audit_rows = []
for raw in raw_web_samples:
    masked = clean_text(mask_pii(raw))
    keep, reason, metrics = quality_filter(raw)
    audit_rows.append({
        "元テキスト": raw[:40] + ("..." if len(raw) > 40 else ""),
        "クレンジング+PIIマスキング後": masked[:52] + ("..." if len(masked) > 52 else ""),
        "日本語率": round(metrics["ja_ratio"], 2),
        "記号率": round(metrics["sym_ratio"], 2),
        "判定": "採用 (KEEP)" if keep else f"除外 ({reason})",
    })
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(audit_rows)

,元テキスト,クレンジング+PIIマスキング後,日本語率,記号率,判定
0,駅近で部屋も清潔でした。問い合わせは support.hotel@example...,駅近で部屋も清潔でした。問い合わせは [EMAIL] または [PHONE] まで。,1.00,0.00,採用 (KEEP)
1,荷物の発送先は 〒191-0065 東京都日野市... 連絡は 090-9876...,荷物の発送先は [POSTAL] 東京都日野市... 連絡は [PHONE] ([USER]) へ!,0.74,0.26,採用 (KEEP)
2,会員ID 1234-5678-9012-3456 で宿泊手続きを完了しました。と...,会員ID [CARD_OR_ID] で宿泊手続きを完了しました。とてもスムーズでした。,0.93,0.00,採用 (KEEP)
3,https://spam-link.example.com/promo?id=9...,Click here!!,0.00,0.18,除外 (low_ja_ratio)
4,Best hotel deal in Tokyo! Book now and g...,Best hotel deal in Tokyo! Book now and get 50% OFF t...,0.00,0.06,除外 (low_ja_ratio)
5,おすすめ★★★★★!!!!!ホテル予約%%%%%今すぐ,おすすめ★★★★★!!!!!ホテル予約%%%%%今すぐ,0.44,0.56,除外 (high_symbol_ratio)
6,okです,okです,0.50,0.00,除外 (too_short)
7,<p>朝食のパンが焼きたてで美味しく、スタッフの笑顔も素敵でした。</p>,朝食のパンが焼きたてで美味しく、スタッフの笑顔も素敵でした。,1.00,0.00,採用 (KEEP)


- **処理順序と境界条件の工夫**: カード番号・電話番号・郵便番号は桁区切りパターンが部分的に重なるため（例：`03-1234-5678` の後半が郵便番号 `XXX-XXXX` と衝突する）、**より長いパターン＋否定後読み・先読み境界 `(?<!\d)...(?!\d)`** で先に `[CARD_OR_ID]` / `[PHONE]` を置換してから `[POSTAL]` を適用しています。また、メールアドレス内の `@example.co.jp` を誤って `[USER]` にしないよう `[EMAIL]` を先に処理しています。
- **ルールベース品質フィルタ**: 言語モデルへ渡す前の段階で、文字長・日本語比率・記号比率の閾値チェックに通すだけで、URL＋英語スパム・記号過多の宣伝文・超短文を一貫して除外できます。